In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv('../data/interim_train.csv')

print(df.shape)
df.head()

(307511, 181)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,EMERGENCYSTATE_MODE_MISSING,OCCUPATION_TYPE_MISSING,EXT_SOURCE_3_MISSING,DAYS_EMPLOYED_MISSING,AMT_REQ_CREDIT_BUREAU_HOUR_MISSING,AMT_REQ_CREDIT_BUREAU_MON_MISSING,AMT_REQ_CREDIT_BUREAU_WEEK_MISSING,AMT_REQ_CREDIT_BUREAU_DAY_MISSING,AMT_REQ_CREDIT_BUREAU_YEAR_MISSING,AMT_REQ_CREDIT_BUREAU_QRT_MISSING
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0,0,0,0,0,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,1,0,0,0,0,0,0,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,1,0,0,0,0,0,0,0,0,0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,1,0,1,0,1,1,1,1,1,1
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,1,0,1,0,0,0,0,0,0,0


In [11]:
# Avoid division by zero
df['AMT_INCOME_TOTAL'] = df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# 1. Credit-to-income ratio
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# 2. Annuity-to-income ratio
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# 3. Credit term proxy
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

# 4. Employment stability
df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']

# 5. Income per family member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

C:\Users\humza\AppData\Local\Temp\ipykernel_25580\2769097063.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
C:\Users\humza\AppData\Local\Temp\ipykernel_25580\2769097063.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
C:\Users\humza\AppData\Local\Temp\ipykernel_25580\2769097063.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.

In [12]:
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

df['EXT_SOURCE_MEAN'] = df[ext_sources].mean(axis=1)
df['EXT_SOURCE_STD'] = df[ext_sources].std(axis=1)

C:\Users\humza\AppData\Local\Temp\ipykernel_25580\3281771827.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['EXT_SOURCE_MEAN'] = df[ext_sources].mean(axis=1)
C:\Users\humza\AppData\Local\Temp\ipykernel_25580\3281771827.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['EXT_SOURCE_STD'] = df[ext_sources].std(axis=1)


In [13]:
df['AGE_BUCKET'] = pd.cut(
    df['age'],
    bins=[20, 30, 40, 50, 60, 70, 100],
    labels=False
)

C:\Users\humza\AppData\Local\Temp\ipykernel_25580\3570389035.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE_BUCKET'] = pd.cut(


In [14]:
bureau = pd.read_csv('../data/bureau.csv')

# Aggregate to customer level
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'CREDIT_ACTIVE': lambda x: (x == 'Active').sum(),
    'DAYS_CREDIT': 'mean',
    'AMT_CREDIT_SUM_OVERDUE': 'max'
}).reset_index()

# Rename columns
bureau_agg.columns = [
    'SK_ID_CURR',
    'num_bureau_records',
    'num_active_credits',
    'mean_days_credit',
    'max_overdue'
]

# Merge
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')

# Fill missing
df[['num_bureau_records', 'num_active_credits']] = df[
    ['num_bureau_records', 'num_active_credits']
].fillna(0)

In [15]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Drop high-cardinality columns (optional, keeps things simple)
high_card_cols = [col for col in cat_cols if df[col].nunique() > 50]
df = df.drop(columns=high_card_cols)

# One-hot encoding
df = pd.get_dummies(df, columns=[col for col in cat_cols if col not in high_card_cols], drop_first=True)

print("New shape after encoding:", df.shape)

C:\Users\humza\AppData\Local\Temp\ipykernel_25580\737155255.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


New shape after encoding: (307511, 244)


In [16]:
print("Missing values left:", df.isna().sum().sum())
df = df.fillna(0)

Missing values left: 88040


In [17]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Drop high-cardinality columns (optional, keeps things simple)
high_card_cols = [col for col in cat_cols if df[col].nunique() > 50]
df = df.drop(columns=high_card_cols)

# One-hot encoding
df = pd.get_dummies(df, columns=[col for col in cat_cols if col not in high_card_cols], drop_first=True)

print("New shape after encoding:", df.shape)

New shape after encoding: (307511, 244)


In [18]:
print("Missing values left:", df.isna().sum().sum())
df = df.fillna(0)

Missing values left: 0


In [19]:
df.to_csv('../data/preprocessed_train.csv', index=False)
print("Saved: preprocessed_train.csv")

Saved: preprocessed_train.csv
